In [1]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib

# ✅ Load the trained model
model_path = "/content/drive/MyDrive/ipl_cricket/final_rf_model.pkl"
model = joblib.load(model_path)

# ✅ Team encoding
team_encoding = {
    'Mumbai Indians': 0, 'Chennai Super Kings': 1, 'Royal Challengers Bengaluru': 2,
    'Kolkata Knight Riders': 3, 'Delhi Capitals': 4, 'Sunrisers Hyderabad': 5,
    'Punjab Kings': 6, 'Rajasthan Royals': 7, 'Gujarat Titans': 8, 'Lucknow Super Giants': 9
}

# ✅ Venue encoding
venue_encoding = {"Wankhede Stadium": 0, "Eden Gardens": 1, "Chidambaram Stadium": 2}

# ✅ Streamlit UI
st.title("🏏 IPL Match Win Predictor")

# Input fields
batting_team = st.selectbox("🏏 Select Batting Team", list(team_encoding.keys()))
bowling_team = st.selectbox("🎯 Select Bowling Team", list(team_encoding.keys()))
venue = st.selectbox("📍 Select Venue", list(venue_encoding.keys()))
total_runs = st.number_input("🏏 Total Runs Scored", min_value=0)
wickets = st.number_input("❌ Wickets Lost", min_value=0, max_value=10)
overs = st.number_input("⏳ Overs Completed", min_value=0.0, max_value=20.0, step=0.1)
is_second_innings = st.checkbox("🌍 Second Innings?")
target = st.number_input("🎯 Target Score", min_value=0) if is_second_innings else 0

# ✅ Calculate CRR & RRR
crr = total_runs / overs if overs > 0 else 0
rrr = ((target - total_runs) / (20 - overs)) if is_second_innings and overs < 20 else 0

st.write(f"📊 *Current Run Rate (CRR):* {crr:.2f}")
if is_second_innings:
    st.write(f"📈 *Required Run Rate (RRR):* {rrr:.2f}")

# ✅ Predict Winner
if st.button("🔮 Predict Winner"):
    input_data = pd.DataFrame([{
        'inning': 2 if is_second_innings else 1,
        'cumulative_runs': total_runs,
        'cumulative_wickets': wickets,
        'current_run_rate': crr,
        'required_run_rate': rrr,
        'target_runs': target,
        'batting_team_encoded': team_encoding[batting_team],
        'bowling_team_encoded': team_encoding[bowling_team],
        'venue_encoded': venue_encoding[venue]
    }])

    prediction = model.predict(input_data)[0]
    winner = batting_team if prediction == 1 else bowling_team
    st.success(f"🏆 **Predicted Winner: {winner}**")


Writing app.py


In [2]:
#!pip install pyngrok
#!ngrok authtoken 2ux2COyjjJYeLyORrCwc3fRQoUZ_7VLRbB4fp3ykpY8ZeB9Xp

In [3]:
# Install required packages
!pip install streamlit pyngrok

# ✅ Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ✅ Step 2: Verify model file existence
import os
import joblib

model_path = "/content/drive/MyDrive/ipl_cricket/final_rf_model.pkl"  # Corrected Path

if os.path.exists(model_path):
    print("✅ File found! Loading the model...")
    model = joblib.load(model_path)
    print("🎉 Model loaded successfully!")
else:
    raise FileNotFoundError(f"❌ Model file not found at {model_path}. Check the path!")

# ✅ Step 3: Setup ngrok authentication
from pyngrok import ngrok

# Uninstall and reinstall pyngrok to fix any issues
!pip uninstall -y pyngrok && pip install pyngrok

# Remove any old ngrok configurations
!rm -rf /root/.ngrok2

# ✅ Set your ngrok authtoken (Replace with your actual token)
NGROK_AUTH_TOKEN = "2ux2COyjjJYeLyORrCwc3fRQoUZ_7VLRbB4fp3ykpY8ZeB9Xp"
os.system(f"ngrok authtoken {NGROK_AUTH_TOKEN}")

# ✅ Restart ngrok before creating a new tunnel
ngrok.kill()

# ✅ Step 4: Start Streamlit app in background
import time
import threading

def run_streamlit():
    os.system("streamlit run app.py --server.port 8501 &")

threading.Thread(target=run_streamlit, daemon=True).start()

# Wait for Streamlit to initialize
time.sleep(5)

# ✅ Step 5: Connect ngrok to Streamlit
public_url = ngrok.connect(8501)
print(f"🚀 Streamlit App is running on {public_url}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 4.9 MB/s eta 0:00:00
Mounted at /content/drive
✅ File found! Loading the model...
🎉 Model loaded successfully!
Found existing installation: pyngrok 7.2.3
Uninstalling pyngrok-7.2.3:
  Successfully uninstalled pyngrok-7.2.3
  Using cached pyngrok-7.2.3-py3-none-any.whl.metadata (8.7 kB)
Using cached pyngrok-7.2.3-py3-none-any.whl (23 kB)
🚀 Streamlit App is running on NgrokTunnel: "https://b45f-34-53-15-36.ngrok-free.app" -> "http://localhost:8501"
